<a href="https://colab.research.google.com/github/Ch-MANASA19/RabTech-KPI-Data-Quality/blob/main/RabTech_Task3_Data_Cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# New Section

In [2]:
import pandas as pd
import numpy as np

# Use the original 12-row dataset as a template
raw = df.copy()

# Generate 12,000 rows for cleaning practice
np.random.seed(42)
large_df = raw.iloc[
    np.random.choice(len(raw), size=12000, replace=True)
].reset_index(drop=True)

# Create unique order IDs
large_df["order_id"] = [
    f"RT-{100001 + i}" for i in range(len(large_df))
]

# Add realistic data quality issues
large_df.loc[large_df.index % 20 == 0, "discount_pct"] = np.nan
large_df.loc[large_df.index % 25 == 0, "city"] = np.nan
large_df.loc[large_df.index % 30 == 0, "quantity"] = np.nan

large_df.loc[large_df.index % 40 == 0, "payment_status"] = "paid"
large_df.loc[large_df.index % 50 == 0, "customer_segment"] = "student"

# Mix date formats
dates = pd.to_datetime(
    large_df["order_date"], dayfirst=True, errors="coerce"
)
large_df["order_date"] = dates.dt.strftime("%Y-%m-%d")
large_df.loc[large_df.index % 10 == 0, "order_date"] = (
    dates.loc[large_df.index % 10 == 0].dt.strftime("%d/%m/%Y")
)

# Introduce duplicate records intentionally
large_df = pd.concat(
    [large_df, large_df.iloc[:100]],
    ignore_index=True
)

large_df.to_csv("retail-orders-messy-12000.csv", index=False)

print("Synthetic messy dataset created!")
print("Rows:", len(large_df))
print("Columns:", len(large_df.columns))
print("Saved as: retail-orders-messy-12000.csv")

Synthetic messy dataset created!
Rows: 12100
Columns: 9
Saved as: retail-orders-messy-12000.csv


In [4]:

import pandas as pd
import numpy as np

# Load the messy dataset
df = pd.read_csv("retail-orders-messy-12000.csv")

print("BEFORE CLEANING")
print("Rows:", len(df))
print("Duplicate rows:", df.duplicated().sum())
print("Missing values:\n", df.isnull().sum())

# 1. Remove exact duplicate records
df = df.drop_duplicates().copy()

# 2. Standardize text values
text_cols = ["customer_segment", "city", "category", "payment_status"]

for col in text_cols:
    df[col] = df[col].astype("string").str.strip()

df["customer_segment"] = df["customer_segment"].str.title()
df["city"] = df["city"].str.title()
df["category"] = df["category"].str.title()
df["payment_status"] = df["payment_status"].str.title()

# 3. Convert data types
df["order_date"] = pd.to_datetime(
    df["order_date"], dayfirst=True, errors="coerce"
)
df["quantity"] = pd.to_numeric(df["quantity"], errors="coerce")
df["unit_price"] = pd.to_numeric(df["unit_price"], errors="coerce")
df["discount_pct"] = pd.to_numeric(df["discount_pct"], errors="coerce")

# 4. Handle invalid values
df.loc[df["quantity"] <= 0, "quantity"] = np.nan
df.loc[df["unit_price"] < 0, "unit_price"] = np.nan
df.loc[~df["discount_pct"].between(0, 100), "discount_pct"] = np.nan

# 5. Impute missing values
df["quantity"] = df["quantity"].fillna(df["quantity"].median())
df["unit_price"] = df["unit_price"].fillna(df["unit_price"].median())
df["discount_pct"] = df["discount_pct"].fillna(0)
df["city"] = df["city"].fillna("Unknown")
df["customer_segment"] = df["customer_segment"].fillna("Unknown")
df["category"] = df["category"].fillna("Unknown")
df["payment_status"] = df["payment_status"].fillna("Unknown")

# 6. Handle missing or invalid dates
df = df.dropna(subset=["order_date"]).copy()

# 7. Feature engineering
df["order_month"] = df["order_date"].dt.month
df["order_year"] = df["order_date"].dt.year

df["gross_sales"] = df["quantity"] * df["unit_price"]
df["net_sales"] = df["gross_sales"] * (
    1 - df["discount_pct"] / 100
)

# Profit margin needs actual cost data.
# Since cost is not available, do not invent profit values.

# 8. Export clean dataset
df.to_csv("clean_dataset.csv", index=False)

print("\nAFTER CLEANING")
print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Remaining missing values:", int(df.isnull().sum().sum()))
print("Remaining exact duplicates:", int(df.duplicated().sum()))
print("\nClean dataset saved as clean_dataset.csv")
print(df.head())


BEFORE CLEANING
Rows: 12100
Duplicate rows: 100
Missing values:
 order_id               0
order_date          5040
customer_segment       0
city                1476
category               0
quantity             404
unit_price             0
discount_pct        1566
payment_status         0
dtype: int64

AFTER CLEANING
Rows: 711
Columns: 13
Remaining missing values: 0
Remaining exact duplicates: 0

Clean dataset saved as clean_dataset.csv
     order_id order_date customer_segment       city        category  \
0   RT-100001 2026-10-13          Student    Unknown    Learning Kit   
20  RT-100021 2026-12-01     Professional     Mumbai  Mentor Session   
40  RT-100041 2026-10-13          Student       Pune    Learning Kit   
60  RT-100061 2026-07-01     Professional  Hyderabad    Learning Kit   
70  RT-100071 2026-12-01     Professional     Mumbai  Mentor Session   

    quantity  unit_price  discount_pct payment_status  order_month  \
0        1.0       799.0           0.0           Paid   

In [5]:

import pandas as pd

raw = pd.read_csv("retail-orders-messy-12000.csv")

# Parse both YYYY-MM-DD and DD/MM/YYYY formats
dates1 = pd.to_datetime(
    raw["order_date"], format="%Y-%m-%d", errors="coerce"
)
dates2 = pd.to_datetime(
    raw["order_date"], format="%d/%m/%Y", errors="coerce"
)

raw["order_date_clean"] = dates1.fillna(dates2)

print("Total rows:", len(raw))
print("Dates parsed successfully:", raw["order_date_clean"].notna().sum())
print("Dates still missing:", raw["order_date_clean"].isna().sum())

print("\nExamples of dates that could not be parsed:")
print(raw.loc[raw["order_date_clean"].isna(), "order_date"].head(10))


Total rows: 12100
Dates parsed successfully: 7060
Dates still missing: 5040

Examples of dates that could not be parsed:
2     NaN
6     NaN
9     NaN
10    NaN
19    NaN
21    NaN
23    NaN
24    NaN
27    NaN
28    NaN
Name: order_date, dtype: object


In [6]:

# Reload the original 12-row dataset
original = pd.read_csv("retail-orders-raw.csv")

# Generate a fresh 12,000-row dataset
import numpy as np

np.random.seed(42)
df = original.iloc[
    np.random.choice(len(original), size=12000, replace=True)
].reset_index(drop=True)

# Assign unique IDs
df["order_id"] = [f"RT-{100001+i}" for i in range(len(df))]

# Parse the original dates safely
df["order_date"] = pd.to_datetime(
    df["order_date"], format="mixed", dayfirst=True, errors="coerce"
)

print("Rows:", len(df))
print("Valid dates:", df["order_date"].notna().sum())
print("Missing dates:", df["order_date"].isna().sum())
print(df[["order_id", "order_date"]].head())


Rows: 12000
Valid dates: 10991
Missing dates: 1009
    order_id order_date
0  RT-100001 2026-10-13
1  RT-100002 2026-01-07
2  RT-100003 2026-01-18
3  RT-100004 2026-01-12
4  RT-100005 2026-01-07


In [7]:

import pandas as pd
import numpy as np

# Work on a copy of the fresh dataset
clean = df.copy()

print("BEFORE CLEANING")
print("Rows:", len(clean))
print("Exact duplicate rows:", clean.duplicated().sum())
print("Missing values:\n", clean.isnull().sum())

# 1. Remove exact duplicate rows
clean = clean.drop_duplicates().copy()

# 2. Standardize text
for col in ["customer_segment", "city", "category", "payment_status"]:
    clean[col] = clean[col].astype("string").str.strip()

clean["customer_segment"] = clean["customer_segment"].str.title()
clean["city"] = clean["city"].str.title()
clean["category"] = clean["category"].str.title()
clean["payment_status"] = clean["payment_status"].str.title()

# 3. Convert numeric columns
for col in ["quantity", "unit_price", "discount_pct"]:
    clean[col] = pd.to_numeric(clean[col], errors="coerce")

# 4. Treat invalid values as missing
clean.loc[clean["quantity"] <= 0, "quantity"] = np.nan
clean.loc[clean["unit_price"] < 0, "unit_price"] = np.nan
clean.loc[~clean["discount_pct"].between(0, 100), "discount_pct"] = np.nan

# 5. Impute numeric values
clean["quantity"] = clean["quantity"].fillna(clean["quantity"].median())
clean["unit_price"] = clean["unit_price"].fillna(clean["unit_price"].median())
clean["discount_pct"] = clean["discount_pct"].fillna(0)

# 6. Fill missing text
for col in ["customer_segment", "city", "category", "payment_status"]:
    clean[col] = clean[col].fillna("Unknown")

# 7. Handle dates
# Use the median valid date only if a date is missing
clean["order_date"] = pd.to_datetime(clean["order_date"], errors="coerce")
median_date = clean["order_date"].dropna().sort_values().iloc[
    len(clean["order_date"].dropna()) // 2
]
clean["order_date"] = clean["order_date"].fillna(median_date)

# 8. Feature engineering
clean["order_month"] = clean["order_date"].dt.month
clean["order_year"] = clean["order_date"].dt.year
clean["gross_sales"] = clean["quantity"] * clean["unit_price"]
clean["net_sales"] = clean["gross_sales"] * (
    1 - clean["discount_pct"] / 100
)

# 9. Export
clean.to_csv("clean_dataset.csv", index=False)

print("\nAFTER CLEANING")
print("Rows:", len(clean))
print("Columns:", len(clean.columns))
print("Missing values:", int(clean.isnull().sum().sum()))
print("Exact duplicate rows:", int(clean.duplicated().sum()))
print("Saved: clean_dataset.csv")
print(clean.head())


BEFORE CLEANING
Rows: 12000
Exact duplicate rows: 0
Missing values:
 order_id               0
order_date          1009
customer_segment       0
city                1024
category               0
quantity               0
unit_price             0
discount_pct        1002
payment_status         0
dtype: int64

AFTER CLEANING
Rows: 12000
Columns: 13
Missing values: 0
Exact duplicate rows: 0
Saved: clean_dataset.csv
    order_id order_date customer_segment       city        category  quantity  \
0  RT-100001 2026-10-13          Student       Pune    Learning Kit       1.0   
1  RT-100002 2026-01-07     Professional  Hyderabad    Learning Kit       3.0   
2  RT-100003 2026-01-18     Professional      Delhi   Course Access       2.0   
3  RT-100004 2026-01-12     Professional     Mumbai  Mentor Session       2.0   
4  RT-100005 2026-01-07     Professional  Hyderabad    Learning Kit       3.0   

   unit_price  discount_pct payment_status  order_month  order_year  \
0       799.0          10.0 

In [8]:

# Before/after cleaning summary

summary = pd.DataFrame({
    "Metric": [
        "Rows",
        "Columns",
        "Missing values",
        "Exact duplicate rows"
    ],
    "Before Cleaning": [
        12000,
        9,
        3035,
        0
    ],
    "After Cleaning": [
        len(clean),
        len(clean.columns),
        int(clean.isnull().sum().sum()),
        int(clean.duplicated().sum())
    ]
})

print(summary.to_string(index=False))
summary.to_csv("cleaning_summary.csv", index=False)


              Metric  Before Cleaning  After Cleaning
                Rows            12000           12000
             Columns                9              13
      Missing values             3035               0
Exact duplicate rows                0               0
